# **Data Centre Digital Twin (DCDT) Simulator: Quick Start Tutorial - Part 2**

This notebook demonstrates how to use the provided Python modules to
simulate a small data hall, control it with either a PID or
reinforcement‑learning(RL) controller, and visualise the results.  It is
intended for working using Google Colab or other Jupyter environments.

**NOTE:** The ***Project Demo Video*** guides through this notebook to get you started, explains how to use this notebook (this notebook uses other modules which are explained more in the ***Project Visual Documentation***) and get this project running!!!

## Install Optional Dependencies

If you wish to train an RL agent with modern algorithms (like PPO), install
[`gymnasium`](https://pypi.org/project/gymnasium/),
[`stable-baselines3`](https://pypi.org/project/stable-baselines3/) and
[`shimmy`](https://pypi.org/project/shimmy/) by running the
following cell.  In Colab you should have a GPU available.

In [1]:
# Run the following line in Colab to install additional packages
!pip install -q stable-baselines3 shimmy gymnasium plotly

## Data Hall Construction for Fault Testing

In [2]:
from grid_editor import create_grid

server_racks = []
cooling_systems = []

## Initialize Data Hall Grid

rows=10                    ## adjust as required
cols=10                    ## adjust as required

server_racks, cooling_systems = create_grid(rows, cols)

Single Click: Server Rack | Double Click: Cooling Unit


## Create the Simulator, Controller and Dashboard for Fault Testing

In [3]:
from simulator import create_default_hall
from controllers import PidController, RLController, ControlSwitch
from dashboard import Dashboard

# Build the data hall
hall_faulty = create_default_hall(server_racks, cooling_systems, rows, cols)

# Create controllers
pid_controller2 = PidController(setpoint=30.0)
rl_controller2 = RLController()

# Wrap both controllers with a switch
controller2 = ControlSwitch(pid_controller=pid_controller2, rl_controller=rl_controller2, use_rl=False)

# Create a dashboard
try:
    dash2 = Dashboard(hall=hall_faulty, controller=controller2)
    print("Systems Initialized.")
except Exception as e:
    print(f"Dashboard could not be initialised: {e}")

[0.3 0.4 0.3 0.3 0.7 0.5 0.7 0.6 0.6 0.3 0.3 0.5 0.6 0.3]
Systems Initialized.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Run the Fault Testing Simulation

In [4]:
from IPython.display import clear_output, display
import time
import random

# Simulation Parameters - Using small dt(keeping below 1.0 is decent) for thermal stability
steps = 200   ## Number of simulation steps
dt_val = 0.1 ## time interval value in seconds

# Fault Fractions
fan_fault_frac = 0.20         ## 20% racks have fan faults
spike_fault_frac = 0.50       ## 50% racks have spike faults
N_fault_fan= int(fan_fault_frac * len(server_racks))
N_fault_spike= int(spike_fault_frac * len(server_racks))

for i in range(steps):
    # Step the physics
    telemetry2 = hall_faulty.step(dt=dt_val)

    # INTERACTIVE NARRATIVE:
    # Switching to RL control halfway to check the shift in efficiency
    if i == int(steps/2):
        controller2.toggle()

    # Fan fault injection
    if i == int(steps/4) or i == int(2*steps/3):
      for i in range(N_fault_fan):
        hall_faulty.inject_fault(random.randint(0, len(server_racks)-1), "fan")

    # Spike fault injection
    if i == int(steps/3) or i == int(3*steps/4):
      for i in range(N_fault_spike):
        hall_faulty.inject_fault(random.randint(0, len(server_racks)-1), "spike")

    # Compute and apply actions
    actions2 = controller2.compute_actions(hall_faulty, telemetry2)
    for idx, signal in actions2.items():
        hall_faulty.cooling_units[idx].control_signal = signal

    # Refresh the UI every 2 steps for smoothness in Colab
    if i % 2 == 0:
        clear_output(wait=True)
        fig2 = dash2.update(telemetry2)
        fig2.show()

        mode_label = "RL (Q-Learning)" if controller2.use_rl else "PID (Baseline)"
        print(f"Step {i+1}/{steps} | Mode: {mode_label}")
        print(f"Time {telemetry2['time']:.1f}s, PUE={telemetry2['pue']:.3f}, MaxTemp={telemetry2['room']['max_temp']:.2f}°C")

    time.sleep(1.0) # Dashboard Update "Frame rate" control

Step 199/200 | Mode: RL (Q-Learning)
Time 19.9s, PUE=1.024, MaxTemp=35.13°C


## Log the Simulation Data (Only for Final Timestep)

In [5]:
from logger import TelemetryLogger
import pandas as pd

# Export final state
logger2 = TelemetryLogger()
logger2.append(telemetry2)
logger2.export_csv("fault_final_telemetry_log.csv")

print("Telemetry successfully exported to fault_final_telemetry_log.csv.")
display(pd.read_csv("fault_final_telemetry_log.csv").head())


Telemetry successfully exported to fault_final_telemetry_log.csv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,time,pue,rack0_id,rack0_position,rack0_cpu_load,rack0_power_draw,rack0_inlet_temp,rack0_exhaust_temp,rack1_id,rack1_position,...,rack13_position,rack13_cpu_load,rack13_power_draw,rack13_inlet_temp,rack13_exhaust_temp,room_average_temp,room_max_temp,room_min_temp,room_humidity,pue.1
0,20.0,1.023464,0,"[5, 4]",0.3,650.0,32.420667,129.920667,1,"[4, 4]",...,"[8, 3]",0.3,650.0,29.006854,94.006854,25.811981,35.168656,18.123116,50.993347,1.023464


## Conclusion

You have successfully run a simple but high-fidelity data‑centre digital twin (DCDT) Simulation, switched between a PID controller and a reinforcement‑learning (RL) controller, and observed how the temperatures and PUE evolved over time along with Power Usage! You have also tried out some basic fault testings, and now in a position to modify, test, and explore yourself!

## Some things to try next:

1.   Try out new grid layout and new coordinates for racks and cooling units.

2.   Try introducing faults with `hall.inject_fault(rack_id, "fan")` or
`hall.inject_fault(rack_id, "spike")` at different timesteps and in different order to see how the system responds.

3.   Try to use fault reset: `hall.inject_fault(rack_id, "spike")` in a meaningful way during the simulation.

4.   For more advanced control, train the RL agent using
`rl_controller.train_stable_baselines(hall, timesteps=50000)` (requires
installation of the optional packages mentioned previously).  

5.   You can also modify the reward function in `controllers.py` to balance energy efficiency
against thermal constraints.

6.   Modify the physics-based considerations.

7.   Modify dashboard to track other parameters.


#### Endless possibilities to try out! Start tinkering! Happy Learning!


**NOTE:** Check the ***Project Visual Documentation*** for more information on the individual modules that this Project uses and how to modify them as per your tests.
